# Sequence Analysis and Exploratory Quality Control

## Research Question
> *"How well do sequence-based predictors of enzymatic polyester degradation generalise to unseen enzyme families and new polymer chemistries?"*

In [3]:
# Environment Setup: compatible with Google Colab and local execution
import sys
import os
from pathlib import Path

if "google.colab" in sys.modules:
    print("Running in Google Colab!")
    if not os.path.exists("src"):
        !git clone https://github.com/zelario/bioinformatics-project.git repo
        %cd repo
    !pip install -q rdkit biopython pandas numpy matplotlib seaborn scikit-learn
else:
    print("Running locally.")

# Add project root to sys.path
project_root = Path(".").resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_pazy_dataset, check_dataset_quality

sns.set_theme(style="whitegrid", font_scale=1.1)
print("✓ Environment ready and modules loaded successfully.")

Running in Google Colab!


ModuleNotFoundError: No module named 'src.data'

## 1. Data Ingestion
We load the verified polyester-active enzymes cataloged in the **PAZy** (Plastics-Active Enzymes) database.

In [ ]:
# Load the PAZy dataset (downloads automatically if not cached locally)
dataset = load_pazy_dataset()
print(f"Total enzymes loaded: {len(dataset)}")
dataset.head(5)

## 2. Quality Control Summary
We verify sequence validity, ensure there are no non-canonical amino acids, inspect sequence lengths, and count verified degradation targets.

In [ ]:
# Compute and display quality control metrics
quality_metrics = check_dataset_quality(dataset)
display(pd.DataFrame(quality_metrics, columns=["Value"]))

## 3. Exploratory Data Visualizations
We inspect the distribution of sequence lengths and the distribution of reported degradation targets (PET, PUR, and PLA).

In [ ]:
# Visualizing sequence length distribution and substrate degradation activity
figure, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sequence length distribution
sns.histplot(dataset["sequence_length"], bins=30, kde=True, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Enzyme Sequence Lengths", fontweight="bold")
axes[0].set_xlabel("Sequence Length (Amino Acids)")
axes[0].set_ylabel("Enzyme Count")

# Plot 2: Substrate activity counts
activity_counts = pd.Series({
    "Active on PET": dataset["active_on_pet"].sum(),
    "Active on PUR": dataset["active_on_pur"].sum(),
    "Active on PLA": dataset["active_on_pla"].sum(),
    "Other Substrates": len(dataset) - dataset["active_on_pet"].sum()
})

activity_counts.plot(kind="bar", ax=axes[1], color=["#e26d5c", "#38b000", "#ffb703", "#8d99ae"])
axes[1].set_title("Substrate Degradation Activity", fontweight="bold")
axes[1].set_ylabel("Enzyme Count")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()